## Import

In [1]:
import pandas as pd
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text

## Tables

In [2]:
SILVER_COMMUNICATIONS = "../../data/silver/communications"
SILVER_CARRIERS       = "../../data/silver/carriers"
SILVER_BROKERS        = "../../data/silver/brokers"

## Dataframes

In [3]:
df_communications = pd.read_parquet(SILVER_COMMUNICATIONS)

df_carrier        = (
    pd.read_parquet(SILVER_CARRIERS)
    .rename(
        columns={
            "id": "carrier_company_id",
            "name": "carrier_name"
        }
    )
)

df_broker         = (
    pd.read_parquet(SILVER_BROKERS)
    .rename(
        columns={
            "id": "broker_company_id",
            "name": "broker_name"
        }
    )
)

In [4]:
df_communications_den = (
    df_communications
    .merge(df_carrier, on="carrier_company_id", how="left")
    .merge(df_broker, on="broker_company_id", how="left")
    [
        [
            "external_id",
            "carrier_company_id",
            "carrier_name",
            "broker_company_id",
            "broker_name",
            "direction",
            "channel",
            "status",
            "created_at",
            "updated_at",
            "from_contact_type",
            "to_contact_type",
            "thread_id"
        ]
    ]
)

In [5]:
outbound = df_communications_den[
    (df_communications_den["direction"] == "outbound")
    & (df_communications_den["status"] == "delivered")
    & (df_communications_den["to_contact_type"].isin(["driver", "dispatcher"]))
].copy()

outbound["contact_type"] = outbound["to_contact_type"]

# outbound.head()

In [6]:
inbound = df_communications_den[
    (df_communications_den["direction"] == "inbound")
    & (df_communications_den["status"] == "received")
    & (df_communications_den["from_contact_type"].isin(["driver", "dispatcher"]))
].copy()

inbound["contact_type"] = inbound["from_contact_type"]

# inbound.head()

In [7]:
append_columns = [
    "external_id",
    "carrier_name",
    "broker_name",
    "direction",
    "channel",
    "contact_type",
    "created_at"
]

events = pd.concat(
    [
        inbound[append_columns],
        outbound[append_columns]
    ],
    ignore_index=True
)

events["is_inbound"] = (
    events["direction"] == "inbound"
).astype(int)

# These are what define a conversation, so we will group by these columns to get the cycle_id
conversation_columns = [
    "external_id",
    "carrier_name",
    "broker_name",
    "channel",
    "contact_type"
]

events = events.sort_values(
    conversation_columns + ["created_at"]
).reset_index(drop=True)

# Here I am creating a cumulative sum based in the inbound message, 
# the ideia is define the cycles of messages inside a conversation, 
# so evertyime when we have a inbound message a new cycle starts
events["cycle_id"] = (
    events
    .groupby(conversation_columns)["is_inbound"]
    .transform(
        lambda x:x.cumsum().shift(fill_value=0)
    )
) 

In [8]:
events["outbound_at"] = events["created_at"].where(
    events["direction"] == "outbound"
)

events["inbound_at"] = events["created_at"].where(
    events["direction"] == "inbound"
)

events["outbound_attempt"] = (
    events["direction"] == "outbound"
).astype(int)

# events.head(50)

In [9]:
responses = (
    events
    .groupby(
        conversation_columns + ["cycle_id"]
        ,as_index=False
    )
    .agg(
        first_outbound_at = ("outbound_at", "min"),
        last_outbound_at = ("outbound_at", "max"),
        response_at = ("inbound_at", "min"),
        outbound_attempts = ("outbound_attempt", "sum")
    )
)

responses["responsed"] = (
    responses["response_at"].notna()
)

responses["response_time"] = (
    (responses["response_at"] - responses["first_outbound_at"]).dt.total_seconds() / 60
)

responses = responses[
    responses["outbound_attempts"] > 0
].copy()

# responses.head(50)

# Save

In [10]:
load_dotenv("../../.env")


database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(database_url)

# with engine.connect() as connection:

#     database = connection.execute(
#         text("SELECT current_database()")
#     ).scalar()

#     print(database)

In [11]:
df_gold = responses.copy()

In [12]:

dataset_name = "carrier_risk"
output_path = f"../../data/gold/{dataset_name}"

# Production S3 example:
# output_path = f"s3://freighthero-data/gold/{dataset_name}"

os.makedirs(
    output_path,
    exist_ok=True
)

df_gold.to_parquet(
    f"{output_path}/data.parquet",
    index=False
)

df_gold.to_sql(
    name=dataset_name,
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

print(
    f"Gold dataset '{dataset_name}' saved successfully."
)

Gold dataset 'carrier_risk' saved successfully.
